# Stage 8a — Download the Validated Covariate Stack

Acquire the additional predictors used by the v2 experiment for the same 2021–2023 period and a source-grid halo around the CHL domain:

- CMEMS surface salinity, surface currents, SSH, mixed-layer depth, and L3 KD490;
- ERA5 hourly 10 m wind components, downloaded in resumable monthly parts and aggregated to daily arithmetic means.

Waves are not part of the current Stage 8 model. Existing files are reused unless their explicit force control is enabled.


## 8a.0 — Configuration

In [13]:
import calendar
from pathlib import Path

import cdsapi
import numpy as np
import pandas as pd
import xarray as xr

BOX = {
    "minimum_longitude": 12.0,
    "maximum_longitude": 15.5,
    "minimum_latitude": 35.5,
    "maximum_latitude": 37.5,
}

# Coarse model grids need a halo so spatial interpolation has source
# cells surrounding every CHL-grid point.
PHY_BOX = {
    "minimum_longitude": 11.9,
    "maximum_longitude": 15.6,
    "minimum_latitude": 35.4,
    "maximum_latitude": 37.6,
}

START = "2021-01-01T00:00:00"
END = "2023-12-31T00:00:00"

RAW_DIR = Path("data/raw")
PROC_DIR = Path("data/processed")

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

CMEMS_JOBS = [
    {
        "dataset_id": "cmems_mod_med_phy-sal_my_4.2km_P1D-m",
        "dataset_version": "202511",
        "dataset_part": "default",
        "variables": ["so"],
        "filename": "phy_sal.nc",
        "label": "salinity",
        "surface_only": True,
        "bbox": PHY_BOX,
    },
    {
        "dataset_id": "cmems_mod_med_phy-cur_my_4.2km_P1D-m",
        "dataset_version": "202511",
        "dataset_part": "default",
        "variables": ["uo", "vo"],
        "filename": "phy_cur.nc",
        "label": "currents",
        "surface_only": True,
        "bbox": PHY_BOX,
    },
    {
        "dataset_id": "cmems_mod_med_phy-ssh_my_4.2km_P1D-m",
        "dataset_version": "202511",
        "dataset_part": "default",
        "variables": ["zos"],
        "filename": "phy_ssh.nc",
        "label": "ssh",
        "surface_only": False,
        "bbox": PHY_BOX,
    },
    {
        "dataset_id": "cmems_mod_med_phy-mld_my_4.2km_P1D-m",
        "dataset_version": "202511",
        "dataset_part": "default",
        "variables": ["mlotst"],
        "filename": "phy_mld.nc",
        "label": "mld",
        "surface_only": False,
        "bbox": PHY_BOX,
    },
    {
        "dataset_id": (
            "cmems_obs-oc_med_bgc-transp_my_l3-multi-1km_P1D"
        ),
        "dataset_version": "202311",
        "dataset_part": "default",
        "variables": ["KD490"],
        "filename": "oc_kd490.nc",
        "label": "kd490",
        "surface_only": False,
        "bbox": BOX,
    },
]

# Keep these labels for the first corrective run. Empty the set after
# the corrected files have been produced.
FORCE_LABELS = set()

ERA5_PARTS_DIR = RAW_DIR / "era5_hourly_parts"
ERA5_PARTS_DIR.mkdir(parents=True, exist_ok=True)

ERA5_PATH = RAW_DIR / "era5_wind_2021_2023.nc"
FORCE_ERA5_PARTS = False
FORCE_ERA5_DAILY = False

ERA5_HOURS = [f"{hour:02d}:00" for hour in range(24)]

print("Target box :", BOX)
print("PHY halo   :", PHY_BOX)
print("Period     :", START, "→", END)

# The four 4.2 km CMEMS fields share a coarser coastline mask.
# Their measured overlap with the finer CHL ocean domain is 96.97%.
PHY_MIN_COVERAGE = 0.969

for job in CMEMS_JOBS:
    depth_text = (
        "surface only"
        if job["surface_only"]
        else "no depth dimension"
    )
    print(
        f"{job['label']:10s} "
        f"version={job['dataset_version']} "
        f"{depth_text} → {job['filename']}"
    )

Target box : {'minimum_longitude': 12.0, 'maximum_longitude': 15.5, 'minimum_latitude': 35.5, 'maximum_latitude': 37.5}
PHY halo   : {'minimum_longitude': 11.9, 'maximum_longitude': 15.6, 'minimum_latitude': 35.4, 'maximum_latitude': 37.6}
Period     : 2021-01-01T00:00:00 → 2023-12-31T00:00:00
salinity   version=202511 surface only → phy_sal.nc
currents   version=202511 surface only → phy_cur.nc
ssh        version=202511 no depth dimension → phy_ssh.nc
mld        version=202511 no depth dimension → phy_mld.nc
kd490      version=202311 no depth dimension → oc_kd490.nc


## 8a.1 — Verify all CMEMS dataset IDs resolve

Fail fast before any long download. If anything errors, run `copernicusmarine describe --contains "<FAMILY>"` (e.g. `MEDSEA_MULTIYEAR_PHY`) and update the `CMEMS_JOBS` table above with the current short-name.

In [14]:
import copernicusmarine

for job in CMEMS_JOBS:
    copernicusmarine.describe(
        dataset_id=job["dataset_id"],
        show_all_versions=True,
        raise_on_error=True,
    )

    print(
        f"OK  {job['label']:10s} "
        f"{job['dataset_id']} "
        f"version={job['dataset_version']}"
    )

Fetching catalogue 1: 100%|██████████| 2/2 [00:00<00:00,  2.15it/s]


OK  salinity   cmems_mod_med_phy-sal_my_4.2km_P1D-m version=202511


Fetching catalogue 1: 100%|██████████| 2/2 [00:00<00:00,  2.41it/s]


OK  currents   cmems_mod_med_phy-cur_my_4.2km_P1D-m version=202511


Fetching catalogue 1: 100%|██████████| 2/2 [00:00<00:00,  2.13it/s]


OK  ssh        cmems_mod_med_phy-ssh_my_4.2km_P1D-m version=202511


Fetching catalogue 1: 100%|██████████| 2/2 [00:00<00:00,  2.28it/s]


OK  mld        cmems_mod_med_phy-mld_my_4.2km_P1D-m version=202511


Fetching catalogue 1:  50%|█████     | 1/2 [00:00<00:00,  1.23it/s]

OK  kd490      cmems_obs-oc_med_bgc-transp_my_l3-multi-1km_P1D version=202311


## 8a.2 — Download CMEMS physics and KD490

Run the pinned job table idempotently. Salinity and currents request only the single model layer between 0 and 2 m; all dynamic products use a spatial halo so interpolation can cover the CHL grid. File size depends on native resolution and variable count, so validation relies on schema, coordinates, time, depth, and measured coverage rather than an estimated download size.


In [15]:
for job in CMEMS_JOBS:
    output_path = RAW_DIR / job["filename"]
    force_this = job["label"] in FORCE_LABELS

    if output_path.exists() and not force_this:
        print(
            f"SKIP  {job['label']:10s} "
            f"{job['filename']} "
            f"({output_path.stat().st_size / 1e6:.1f} MB)"
        )
        continue

    request = {
        "dataset_id": job["dataset_id"],
        "dataset_version": job["dataset_version"],
        "dataset_part": job["dataset_part"],
        "variables": job["variables"],
        "start_datetime": START,
        "end_datetime": END,
        "output_filename": job["filename"],
        "output_directory": str(RAW_DIR),
        "coordinates_selection_method": "inside",
        "overwrite": force_this,
        **job["bbox"],
    }

    # The first model depth is approximately 1.018 m.
    # Selecting 0–2 m retains exactly that surface layer.
    if job["surface_only"]:
        request["minimum_depth"] = 0.0
        request["maximum_depth"] = 2.0

    print(
        f"\nDownloading {job['label']} "
        f"from version {job['dataset_version']}..."
    )

    copernicusmarine.subset(**request)

    if not output_path.exists():
        raise RuntimeError(
            f"Download did not create {output_path}"
        )

    print(
        f"OK  {job['label']:10s} → {output_path} "
        f"({output_path.stat().st_size / 1e6:.1f} MB)"
    )

SKIP  salinity   phy_sal.nc (20.3 MB)
SKIP  currents   phy_cur.nc (40.6 MB)
SKIP  ssh        phy_ssh.nc (20.3 MB)
SKIP  mld        phy_mld.nc (20.3 MB)
SKIP  kd490      oc_kd490.nc (235.9 MB)


## 8a.3 — ERA5 wind via CDS

Configure CDS credentials once, then request all 24 hourly `u10` and `v10` values in one file per calendar month. Monthly requests are small enough to avoid the CDS cost-limit failure produced by a single three-year request and can be resumed independently. Each part is validated against its complete hourly sequence before reuse; the final file is the daily arithmetic mean with an exact 1,095-day axis.

`FORCE_ERA5_PARTS` controls deliberate replacement of monthly inputs, while `FORCE_ERA5_DAILY` controls rebuilding the daily aggregate.


In [16]:
ERA5_DATASET = "reanalysis-era5-single-levels"
ERA5_VARIABLES = [
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
]

# CDS area order: North, West, South, East
ERA5_AREA = [37.5, 12.0, 35.5, 15.5]


def era5_time_name(ds):
    for name in ("valid_time", "time"):
        if name in ds.coords or name in ds.dims:
            return name
    raise ValueError("ERA5 file has no valid_time or time coordinate")


def expected_monthly_hours(year, month):
    n_days = calendar.monthrange(year, month)[1]
    start = pd.Timestamp(year=year, month=month, day=1)
    return pd.date_range(start, periods=n_days * 24, freq="h")


def era5_part_is_valid(path, year, month):
    if not path.exists():
        return False

    try:
        with xr.open_dataset(path) as ds:
            time_name = era5_time_name(ds)
            actual = pd.DatetimeIndex(ds[time_name].values)
            expected = expected_monthly_hours(year, month)

            return (
                "u10" in ds.data_vars
                and "v10" in ds.data_vars
                and not actual.has_duplicates
                and actual.equals(expected)
            )
    except Exception:
        return False


client = cdsapi.Client()
era5_part_paths = []

for year in range(2021, 2024):
    for month in range(1, 13):
        part_path = (
            ERA5_PARTS_DIR
            / f"era5_wind_{year}_{month:02d}.nc"
        )
        era5_part_paths.append(part_path)

        if (
            not FORCE_ERA5_PARTS
            and era5_part_is_valid(part_path, year, month)
        ):
            print(f"✓ Reusing {part_path.name}")
            continue

        n_days = calendar.monthrange(year, month)[1]

        request = {
            "product_type": ["reanalysis"],
            "variable": ERA5_VARIABLES,
            "year": [str(year)],
            "month": [f"{month:02d}"],
            "day": [
                f"{day:02d}"
                for day in range(1, n_days + 1)
            ],
            "time": ERA5_HOURS,
            "area": ERA5_AREA,
            "data_format": "netcdf",
            "download_format": "unarchived",
        }

        temporary_path = part_path.with_name(
            part_path.stem + ".partial.nc"
        )

        if temporary_path.exists():
            temporary_path.unlink()

        print(f"↓ Downloading {year}-{month:02d}")
        result = client.retrieve(ERA5_DATASET, request)
        result.download(str(temporary_path))

        if not era5_part_is_valid(temporary_path, year, month):
            raise RuntimeError(
                f"Downloaded ERA5 file failed validation: "
                f"{temporary_path}"
            )

        temporary_path.replace(part_path)
        print(f"✓ Saved {part_path.name}")

✓ Reusing era5_wind_2021_01.nc
✓ Reusing era5_wind_2021_02.nc
✓ Reusing era5_wind_2021_03.nc
✓ Reusing era5_wind_2021_04.nc
✓ Reusing era5_wind_2021_05.nc
✓ Reusing era5_wind_2021_06.nc
✓ Reusing era5_wind_2021_07.nc
✓ Reusing era5_wind_2021_08.nc
✓ Reusing era5_wind_2021_09.nc
✓ Reusing era5_wind_2021_10.nc
✓ Reusing era5_wind_2021_11.nc
✓ Reusing era5_wind_2021_12.nc
✓ Reusing era5_wind_2022_01.nc
✓ Reusing era5_wind_2022_02.nc
✓ Reusing era5_wind_2022_03.nc
✓ Reusing era5_wind_2022_04.nc
✓ Reusing era5_wind_2022_05.nc
✓ Reusing era5_wind_2022_06.nc
✓ Reusing era5_wind_2022_07.nc
✓ Reusing era5_wind_2022_08.nc
✓ Reusing era5_wind_2022_09.nc
✓ Reusing era5_wind_2022_10.nc
✓ Reusing era5_wind_2022_11.nc
✓ Reusing era5_wind_2022_12.nc
✓ Reusing era5_wind_2023_01.nc
✓ Reusing era5_wind_2023_02.nc
✓ Reusing era5_wind_2023_03.nc
✓ Reusing era5_wind_2023_04.nc
✓ Reusing era5_wind_2023_05.nc
✓ Reusing era5_wind_2023_06.nc
✓ Reusing era5_wind_2023_07.nc
✓ Reusing era5_wind_2023_08.nc
✓ Reusin

In [17]:
expected_hourly = pd.date_range(
    "2021-01-01 00:00:00",
    "2023-12-31 23:00:00",
    freq="h",
)

expected_daily = pd.date_range(
    "2021-01-01",
    "2023-12-31",
    freq="D",
)


def era5_daily_is_valid(path):
    if not path.exists():
        return False

    try:
        with xr.open_dataset(path) as ds:
            time_name = era5_time_name(ds)
            actual = pd.DatetimeIndex(ds[time_name].values)

            return (
                "u10" in ds.data_vars
                and "v10" in ds.data_vars
                and not actual.has_duplicates
                and actual.equals(expected_daily)
            )
    except Exception:
        return False


if FORCE_ERA5_DAILY or not era5_daily_is_valid(ERA5_PATH):
    opened_datasets = []

    try:
        for path in era5_part_paths:
            if not path.exists():
                raise FileNotFoundError(
                    f"Missing ERA5 monthly file: {path}"
                )

            ds = xr.open_dataset(path)

            if "valid_time" in ds.coords:
                ds = ds.rename({"valid_time": "time"})

            # Remove singleton CDS metadata dimensions/coordinates.
            for nuisance in ("number", "expver"):
                if nuisance in ds.dims:
                    if ds.sizes[nuisance] != 1:
                        raise ValueError(
                            f"{path.name}: unexpected "
                            f"{nuisance} size {ds.sizes[nuisance]}"
                        )
                    ds = ds.isel({nuisance: 0}, drop=True)
                elif nuisance in ds.variables:
                    ds = ds.drop_vars(nuisance)

            ds = ds[["u10", "v10"]]
            opened_datasets.append(ds)

        hourly = xr.concat(
            opened_datasets,
            dim="time",
            data_vars="minimal",
            coords="minimal",
            compat="override",
            join="exact",
        ).sortby("time")

        actual_hourly = pd.DatetimeIndex(hourly["time"].values)

        missing = expected_hourly.difference(actual_hourly)
        extra = actual_hourly.difference(expected_hourly)

        if (
            actual_hourly.has_duplicates
            or not actual_hourly.equals(expected_hourly)
        ):
            raise RuntimeError(
                "ERA5 hourly timeline is incomplete: "
                f"rows={len(actual_hourly)}, "
                f"duplicates={actual_hourly.duplicated().sum()}, "
                f"missing={len(missing)}, extra={len(extra)}"
            )

        daily = hourly.resample(time="1D").mean(
            skipna=True,
            keep_attrs=True,
        )

        # Use the same coordinate orientation as the other covariates.
        if (
            "latitude" in daily.coords
            and daily.latitude.values[0] > daily.latitude.values[-1]
        ):
            daily = daily.sortby("latitude")

        actual_daily = pd.DatetimeIndex(daily["time"].values)

        if not actual_daily.equals(expected_daily):
            raise RuntimeError(
                f"Expected {len(expected_daily)} daily timestamps, "
                f"found {len(actual_daily)}"
            )

        daily = daily.astype(np.float32).load()

        daily.attrs.update({
            "title": "ERA5 daily mean 10 m wind components",
            "temporal_aggregation": (
                "Arithmetic mean of all 24 hourly values per UTC day"
            ),
            "source_dataset": ERA5_DATASET,
        })

        temporary_daily = ERA5_PATH.with_name(
            ERA5_PATH.stem + ".partial.nc"
        )

        if temporary_daily.exists():
            temporary_daily.unlink()

        daily.to_netcdf(
            temporary_daily,
            encoding={
                "u10": {"dtype": "float32"},
                "v10": {"dtype": "float32"},
            },
        )

        temporary_daily.replace(ERA5_PATH)

    finally:
        for ds in opened_datasets:
            ds.close()

    print(f"✓ Daily ERA5 file written: {ERA5_PATH}")
else:
    print(f"✓ Reusing valid daily ERA5 file: {ERA5_PATH}")


with xr.open_dataset(ERA5_PATH) as check:
    print(check)
    print(
        "Dates:",
        check.time.values[0],
        "to",
        check.time.values[-1],
    )

✓ Reusing valid daily ERA5 file: data\raw\era5_wind_2021_2023.nc
<xarray.Dataset> Size: 1MB
Dimensions:    (time: 1095, latitude: 9, longitude: 15)
Coordinates:
  * time       (time) datetime64[ns] 9kB 2021-01-01 2021-01-02 ... 2023-12-31
  * latitude   (latitude) float64 72B 35.5 35.75 36.0 36.25 ... 37.0 37.25 37.5
  * longitude  (longitude) float64 120B 12.0 12.25 12.5 ... 15.0 15.25 15.5
Data variables:
    u10        (time, latitude, longitude) float32 591kB ...
    v10        (time, latitude, longitude) float32 591kB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-08-20T14:43 GRIB to CDM+CF via cfgrib-0.9.1...
    title:                   ERA5 daily mean 10 m wind components
    temporal_aggregation:    Arithmetic mean of

---
## Validation Gate 8a

Every CMEMS file and the ERA5 daily file are required. The gate verifies expected variables, exact daily time axes, ascending coordinates, source grids that surround CHL, and exactly one 0–2 m layer for surface-only products. Surface physics must cover at least 96.9% of the observable CHL domain on the representative day; KD490 must be non-empty while retaining genuine L3 gaps. ERA5 must contain finite daily `u10`/`v10` values with no hourly dimension left.

The measured coverages and pinned acquisition metadata are saved to `covariate_acquisition_manifest.json`.


In [18]:
import json
import numpy as np
import pandas as pd
import xarray as xr

CHL_PATH = RAW_DIR / "chl_malta_2021_2023.nc"
DOMAIN_PATH = PROC_DIR / "chl_target_domain.nc"

assert CHL_PATH.exists()
assert DOMAIN_PATH.exists()

chl = xr.open_dataset(CHL_PATH)["CHL"]
target_domain = (
    xr.open_dataarray(DOMAIN_PATH)
    .astype(bool)
    .load()
)

expected_time = pd.date_range(
    pd.Timestamp(START),
    pd.Timestamp(END),
    freq="D",
)

results = {}
coverage_report = {}

def strictly_ascending(values):
    return bool(
        np.all(np.diff(np.asarray(values)) > 0)
    )

for job in CMEMS_JOBS:
    path = RAW_DIR / job["filename"]
    label = job["label"]

    results[f"{label}: file present"] = (
        path.exists()
        and path.stat().st_size > 100_000
    )

    if not path.exists():
        continue

    ds = xr.open_dataset(
        path,
        chunks={"time": 31},
    )

    results[f"{label}: variables present"] = (
        set(job["variables"])
        .issubset(set(ds.data_vars))
    )

    actual_time = pd.DatetimeIndex(ds.time.values)

    results[f"{label}: exact daily time axis"] = (
        actual_time.equals(expected_time)
    )

    results[f"{label}: axes are ascending"] = (
        strictly_ascending(ds.latitude.values)
        and strictly_ascending(ds.longitude.values)
    )

    results[f"{label}: grid surrounds CHL"] = (
        float(ds.longitude.min())
        <= float(chl.longitude.min())
        and float(ds.longitude.max())
        >= float(chl.longitude.max())
        and float(ds.latitude.min())
        <= float(chl.latitude.min())
        and float(ds.latitude.max())
        >= float(chl.latitude.max())
    )

    if job["surface_only"]:
        results[f"{label}: exactly one surface depth"] = (
            ds.sizes.get("depth") == 1
            and 0.0 < float(ds.depth.values[0]) < 2.0
        )

    variable = ds[job["variables"][0]]

    if "depth" in variable.dims:
        variable = variable.isel(depth=0)

    representative = (
        variable
        .isel(time=0)
        .interp(
            latitude=chl.latitude,
            longitude=chl.longitude,
        )
        .compute()
    )

    valid_on_domain = (
        representative.notnull()
        & target_domain
    )

    domain_coverage = (
        float(valid_on_domain.sum())
        / float(target_domain.sum())
    )
    coverage_report[label] = domain_coverage

    if label in {"salinity", "currents", "ssh", "mld"}:
    # Residual coastal gaps are expected because the 4.2 km
    # model coastline is coarser than the CHL coastline.
        coverage_check = (
            f"{label}: ≥{PHY_MIN_COVERAGE:.1%} "
            "CHL-domain coverage"
        )
        results[coverage_check] = (
            domain_coverage >= PHY_MIN_COVERAGE
        )
    else:
        # KD490 is L3 ocean colour and should retain genuine cloud gaps.
        results["kd490: representative day is nonempty"] = (
            int(valid_on_domain.sum()) > 0
        )
        results["kd490: representative day retains gaps"] = (
            domain_coverage < 0.95
        )

    ds.close()

# ERA5 daily product
results["era5: daily file present"] = (
    ERA5_PATH.exists()
    and ERA5_PATH.stat().st_size > 100_000
)

if ERA5_PATH.exists():
    era5 = xr.open_dataset(ERA5_PATH)

    results["era5: variables present"] = (
        {"u10", "v10"}
        .issubset(set(era5.data_vars))
    )

    results["era5: no hourly dimension remains"] = (
        era5.sizes.get("time") == 1095
    )

    era5_time = pd.DatetimeIndex(era5.time.values)

    results["era5: exact daily time axis"] = (
        era5_time.equals(expected_time)
    )

    results["era5: grid surrounds CHL"] = (
        float(era5.longitude.min())
        <= float(chl.longitude.min())
        and float(era5.longitude.max())
        >= float(chl.longitude.max())
        and float(era5.latitude.min())
        <= float(chl.latitude.min())
        and float(era5.latitude.max())
        >= float(chl.latitude.max())
    )

    results["era5: axes are ascending"] = (
        strictly_ascending(era5.latitude.values)
        and strictly_ascending(era5.longitude.values)
    )

    results["era5: daily values are finite"] = bool(
        era5[["u10", "v10"]]
        .to_array()
        .notnull()
        .all()
        .item()
    )

    era5.close()

for name, ok in results.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")

print("\nRepresentative CHL-domain coverage:")

for label, coverage in coverage_report.items():
    print(f"  {label:10s}: {coverage * 100:.2f}%")

failed = [
    name
    for name, ok in results.items()
    if not ok
]

if failed:
    raise RuntimeError(
        "Validation Gate 8a failed:\n- "
        + "\n- ".join(failed)
    )

acquisition_manifest = {
    "study_start": START,
    "study_end": END,
    "target_box": BOX,
    "phy_download_box": PHY_BOX,
    "cmems_jobs": [
        {
            key: value
            for key, value in job.items()
            if key != "bbox"
        }
        for job in CMEMS_JOBS
    ],
    "era5": {
        "source_dataset": "reanalysis-era5-single-levels",
        "raw_frequency": "hourly",
        "downloaded_hours_per_day": 24,
        "aggregation": "daily arithmetic mean",
        "hourly_parts_directory": str(ERA5_PARTS_DIR),
        "hourly_part_count": len(
            list(
                ERA5_PARTS_DIR.glob("era5_wind_????_??.nc")
            )
        ),
        "daily_file": str(ERA5_PATH),
    },
    "representative_domain_coverage": (
        coverage_report
    ),
}

manifest_path = (
    PROC_DIR / "covariate_acquisition_manifest.json"
)
manifest_path.write_text(
    json.dumps(acquisition_manifest, indent=2)
)

print("\nGate 8a PASSED.")
print(f"Manifest saved → {manifest_path}")

Fetching catalogue 1: 100%|██████████| 2/2 [00:02<00:00,  1.44s/it]

PASS  salinity: file present
PASS  salinity: variables present
PASS  salinity: exact daily time axis
PASS  salinity: axes are ascending
PASS  salinity: grid surrounds CHL
PASS  salinity: exactly one surface depth
PASS  salinity: ≥96.9% CHL-domain coverage
PASS  currents: file present
PASS  currents: variables present
PASS  currents: exact daily time axis
PASS  currents: axes are ascending
PASS  currents: grid surrounds CHL
PASS  currents: exactly one surface depth
PASS  currents: ≥96.9% CHL-domain coverage
PASS  ssh: file present
PASS  ssh: variables present
PASS  ssh: exact daily time axis
PASS  ssh: axes are ascending
PASS  ssh: grid surrounds CHL
PASS  ssh: ≥96.9% CHL-domain coverage
PASS  mld: file present
PASS  mld: variables present
PASS  mld: exact daily time axis
PASS  mld: axes are ascending
PASS  mld: grid surrounds CHL
PASS  mld: ≥96.9% CHL-domain coverage
PASS  kd490: file present
PASS  kd490: variables present
PASS  kd490: exact daily time axis
PASS  kd490: axes are ascend

---
## Next: `08b-Group-Importance-Ablation.ipynb`

Notebook 8b interpolates the validated sources onto the CHL grid, adds salinity, currents/SSH, MLD, KD490, and wind-derived features to the exact Notebook 4 sample keys, and evaluates eight feature groups. Wind diagnostics are speed, 3- and 7-day speed means, a wind-stress-curl proxy, and the two daily wind components. No Bakun or wave feature is used in the current model.
